In [1]:
import geopy.distance
import pandas as pd
import numpy as np
from glob import glob
from os.path import join
from multiprocessing import Pool
import pyarrow.parquet as pq
from skmob.core.trajectorydataframe import *
from skmob.measures import individual
from skmob.preprocessing import filtering, compression, detection, clustering
from skmob.utils import utils, constants
from src.clustering_st import cluster
import folium

In [2]:
base_data_path = "/glade/derecho/scratch/cbecker/CAT_daily/chunked_data/"
key_file = "/glade/derecho/scratch/cbecker/CAT_daily/chunked_data/id_row_group_key.parquet"
daily_files = sorted(glob(join(base_data_path, "veraset*")))

In [3]:
## load the metadata
meta = [pq.ParquetFile(x).metadata for x in daily_files]

In [4]:
def get_row_groups(row_group, files):
    """ Read a specific row group across all files. """
    
    return pd.concat([pq.ParquetFile(f, metadata=m).read_row_group(row_group).to_pandas() for f, m in zip(files, meta)]) 


def get_row_group_trajectories(row_group, files):
    """ Read the row group from all files and return a skmobility Trajectory DataFrame """
    df = get_row_groups(row_group, files)
    tdf = TrajDataFrame(df.reset_index(), latitude='latitude', longitude='longitude', datetime='datetime', user_id='caid')
    
    return tdf

In [5]:
row_groups = np.arange(2975, 2976)
files = [daily_files] * len(row_groups)
args = [(rg, f) for rg, f in zip(row_groups, files)]

In [6]:
with Pool(processes=8) as pool:
    data = pd.concat(pool.starmap(get_row_group_trajectories, args))

In [7]:
data

,uid,datetime,id_type,lat,lng,horizontal_accuracy
0,0d451a45e0cb218279eeb303725bd60dd602a3120dc87e...,2024-03-14 23:00:15-07:00,aaid,38.625912,-121.263733,23
1,0d451a45e0cb218279eeb303725bd60dd602a3120dc87e...,2024-03-14 23:00:16-07:00,aaid,38.625950,-121.263718,23
2,0d451a45e0cb218279eeb303725bd60dd602a3120dc87e...,2024-03-14 23:02:54-07:00,aaid,38.625061,-121.263603,20
3,0d451a45e0cb218279eeb303725bd60dd602a3120dc87e...,2024-03-14 23:03:01-07:00,aaid,38.625069,-121.263603,20
4,0d451a45e0cb218279eeb303725bd60dd602a3120dc87e...,2024-03-14 23:03:02-07:00,aaid,38.625069,-121.263603,20
...,...,...,...,...,...,...
87969,0d44e2b7009e4bafe0592a7f925a2880b4b55cae07d4fd...,2024-04-30 14:34:20-07:00,aaid,34.047569,-117.916962,8
87970,0d44e2b7009e4bafe0592a7f925a2880b4b55cae07d4fd...,2024-04-30 15:01:14-07:00,aaid,34.047630,-117.917122,6
87971,0d44af1a095316d5713709569a95606ea97f3cd769790b...,2024-04-30 11:38:50-07:00,aaid,34.113651,-118.073334,5
87972,0d44af1a095316d5713709569a95606ea97f3cd769790b...,2024-04-30 11:38:51-07:00,aaid,34.113201,-118.067307,5


In [8]:
uid = "0d449a0b33c1803ce62b6eab05ea696fc9008a77a9e2e37f6a9146ac18556757"
data_uid = data[data[constants.UID] == uid].drop("uid", axis=1)
data_uid

,datetime,id_type,lat,lng,horizontal_accuracy
1589,2024-03-15 04:38:40-07:00,idfa,40.770531,-124.191589,3
1590,2024-03-15 04:38:43-07:00,idfa,40.770561,-124.191498,3
1591,2024-03-15 04:38:46-07:00,idfa,40.770512,-124.191498,3
1592,2024-03-15 04:38:47-07:00,idfa,40.770512,-124.191513,3
1593,2024-03-15 04:41:11-07:00,idfa,40.761440,-124.190338,12
...,...,...,...,...,...
76195,2024-04-26 08:05:59-07:00,idfa,38.891029,-123.053642,5
76196,2024-04-26 08:33:20-07:00,idfa,38.450600,-122.726196,11
76197,2024-04-26 08:33:23-07:00,idfa,38.450600,-122.726250,11
76198,2024-04-26 08:34:31-07:00,idfa,38.436550,-122.717506,21


In [9]:
print(data_uid.shape)
data_uid.plot_trajectory(zoom=6)

(145, 5)


/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: Only the trajectories of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_trajectory(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: If necessary, trajectories will be down-sampled to have at most `max_points` points. To avoid this, specify `max_points=None`.
  return plot.plot_trajectory(


![title](img/hl_byzip_bytime.png)

In [10]:
data_uid.plot_stops(zoom=6)

/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:791: UserWarning: Only the stops of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_stops(


In [11]:
m = folium.Map(location=[34.0549, -118.2426],
                    zoom_start=5,
                    control_scale=True,
              )
folium.CircleMarker(location=[32.701630, -116.355110],
                    radius=6,
                    weight=5,
                    zoom=6,
                    tooltip="TAO").add_to(m)
folium.CircleMarker(location=[41.747719, -124.202972],
                    radius=6,
                    weight=5,
                    zoom=6,
                    tooltip="Original").add_to(m)
folium.CircleMarker(location=[32.848270, -116.954502],
                    radius=6,
                    weight=5,
                    zoom=6,
                    tooltip="TAO Work").add_to(m)
m

In [12]:
def filter_by_min_pings_per_daterange(tdf, start_date='2024-03-14', end_date='2024-04-30', min_pings=1):

    # filter by date range, inclusive    
    _tdf = tdf.copy(deep=True)
    _tdf.loc[:, constants.DATETIME] = _tdf[constants.DATETIME].dt.normalize()
    _tdf = _tdf[(_tdf[constants.DATETIME] >= start_date) & (_tdf[constants.DATETIME] <= end_date)]
    threshold = (len(_tdf[constants.DATETIME].unique()) - 1) * min_pings
    
    # if 'uid' column in not present in the TrajDataFrame
    if constants.UID not in _tdf.columns:
        _tdf = _tdf.set_index(constants.DATETIME, drop=True)
        _tdf = _tdf.loc[:, constants.LATITUDE].groupby(level=0).count()
        if len(_tdf[_tdf > 1]) > threshold:
            return tdf
    else:
        _tdf = _tdf.set_index([constants.UID, constants.DATETIME], drop=True)
        _tdf = _tdf.loc[:, constants.LATITUDE].groupby(level=[0,1]).count()
        _tdf[_tdf > 1] = 1
        _tdf = _tdf.groupby(level=0).count()
        uids = _tdf[_tdf > threshold].index
        
        return tdf[tdf[constants.UID].isin(uids)]

In [13]:
traj = filter_by_min_pings_per_daterange(data_uid, start_date='2024-03-28', end_date='2024-03-30', min_pings=1)
print(traj.shape)
traj.plot_trajectory(zoom=6)

(145, 5)


/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: Only the trajectories of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_trajectory(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: If necessary, trajectories will be down-sampled to have at most `max_points` points. To avoid this, specify `max_points=None`.
  return plot.plot_trajectory(


In [14]:
traj.plot_stops(zoom=6)

/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:791: UserWarning: Only the stops of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_stops(


In [15]:
def filter_by_horizontal_accuracy_and_speed(tdf, max_horizontal_accuracy=800, max_speed_kmh=128.7):

    columns = [constants.LATITUDE, constants.LONGITUDE]
    distfunc = geopy.distance.distance
    
    tdf.reset_index(inplace=True)
    for i in tdf[tdf.horizontal_accuracy >= max_horizontal_accuracy].index:
        if i-1 in tdf.index and tdf.loc[i-1, "uid"] == tdf.loc[i, "uid"]:
            start = i - 1
            end = i
        elif i+1 in tdf.index and tdf.loc[i+1, "uid"] == tdf.loc[i, "uid"]:
            start = i
            end = i + 1
        else:
            # print(f"skipped and condition: {i-1 in tdf.index} and {tdf.loc[i-1, 'uid'] == tdf.loc[i, 'uid']} and {tdf.loc[i, 'horizontal_accuracy']}")
            continue
        
        dt = utils.diff_seconds(tdf.loc[start, constants.DATETIME], tdf.loc[end, constants.DATETIME])
        distance = distfunc(tdf.loc[start, columns].tolist(), tdf.loc[end, columns].tolist()).km
        
        if distance / dt * 3600. > max_speed_kmh:
            tdf = tdf.drop(i)
            # print("REJECTED", i, speed)
    
    return tdf.set_index("index", drop=True)

In [16]:
traj = filter_by_horizontal_accuracy_and_speed(traj, max_horizontal_accuracy=800, max_speed_kmh=128.7)
print(traj.shape)
traj.plot_trajectory(zoom=6)

(145, 5)


/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: Only the trajectories of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_trajectory(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: If necessary, trajectories will be down-sampled to have at most `max_points` points. To avoid this, specify `max_points=None`.
  return plot.plot_trajectory(


In [17]:
traj.plot_stops(zoom=6)

/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:791: UserWarning: Only the stops of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_stops(


In [18]:
traj_noloops = filtering.filter(traj, max_speed_kmh=10000.0)
print(traj_noloops.shape)
traj_noloops = filtering.filter(traj_noloops, max_speed_kmh=10000.0, include_loops=True, speed_kmh=5.0, max_loop=6, ratio_max=0.25)
print(traj_noloops.shape)
traj_noloops.plot_trajectory(zoom=6)

(145, 5)
(143, 5)


/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: Only the trajectories of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_trajectory(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: If necessary, trajectories will be down-sampled to have at most `max_points` points. To avoid this, specify `max_points=None`.
  return plot.plot_trajectory(


In [19]:
traj_noloops.plot_stops(zoom=6)

/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:791: UserWarning: Only the stops of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_stops(


In [20]:
traj_noloops.plot_stops(zoom=6)

/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:791: UserWarning: Only the stops of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_stops(


In [21]:
traj_noloops_maxspeed = filtering.filter(traj_noloops, max_speed_kmh=1000.0)
print(traj_noloops_maxspeed.shape)
traj_noloops_maxspeed.plot_trajectory(zoom=6)

(142, 5)


/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: Only the trajectories of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_trajectory(
/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:689: UserWarning: If necessary, trajectories will be down-sampled to have at most `max_points` points. To avoid this, specify `max_points=None`.
  return plot.plot_trajectory(


In [22]:
traj_noloops_maxspeed.plot_stops(zoom=6)

/glade/work/ggantos/conda-envs/cat/lib/python3.11/site-packages/skmob/core/trajectorydataframe.py:791: UserWarning: Only the stops of the first 10 users will be plotted. Use the argument `max_users` to specify the desired number of users, or filter the TrajDataFrame.
  return plot.plot_stops(


In [29]:
if ("uid" in data.index) or ("uid" in data.columns):
    print("yes")

yes


In [30]:
if ("uid" in traj_noloops_maxspeed.index) or ("uid" in traj_noloops_maxspeed.columns):
    print("yes")

In [49]:
def home_location_zao(tdf, **kwargs):

    _tdf = detection.stay_locations(tdf,
                                    stop_radius_factor=0.5,
                                    minutes_for_a_stop=10.0,
                                    spatial_radius_km=0.2,
                                    leaving_time=True,
                                    no_data_for_minutes=1000000000000.0,
                                    min_speed_kmh=None)
    _tdf = clustering.cluster(_tdf, cluster_radius_km=0.1, min_samples=1)
    _tdf["duration"] = _tdf["leaving_datetime"] - _tdf["datetime"]
    
    if ("uid" in _tdf.index) or ("uid" in _tdf.columns):
        _tdf1 = _tdf.groupby(["uid", "cluster"])["duration"].sum()
        _tdf2 = _tdf.groupby(["uid", "cluster"])[["lat", "lng"]].mean()
        _tdfhw = pd.concat((_tdf1.to_frame(), _tdf2), axis=1, join="inner")
        _tdfhw = _tdfhw.sort_values(by=['uid', 'duration'], ascending=[False, False])
        _tdfhw = _tdfhw.groupby("uid").head(2)
    
        home = _tdfhw.groupby("uid").head(1).droplevel(1)
        home = home.rename(columns={"duration": "duration_home"})
        work = _tdfhw.groupby("uid").tail(1).droplevel(1)
        work = work.rename(columns={"lat": "lat_work", "lng": "lng_work", "duration": "duration_work"})

        return home.merge(work, left_index=True, right_index=True)
    
    else:
        _tdf1 = _tdf.groupby("cluster")["duration"].sum()
        _tdf2 = _tdf.groupby("cluster")[["lat", "lng"]].mean()
        _tdfhw = pd.concat((_tdf1.to_frame(), _tdf2), axis=1, join="inner")
        home =  _tdfhw.loc[0].rename({"duration": "duration_home"})
        work = _tdfhw.loc[1].rename({"lat": "lat_work", "lng": "lng_work", "duration": "duration_work"})
    
        return pd.concat([home, work], axis=0)

In [50]:
home_locs = home_location_zao(traj_noloops_maxspeed)
home_locs

/glade/derecho/scratch/ggantos/tmp/ipykernel_53292/4015582592.py:10: DeprecationWarning: This method will be deprecated in version 2.0. Instead, use the dbscan() method.
  _tdf = clustering.cluster(_tdf, cluster_radius_km=0.1, min_samples=1)


duration_home    0 days 23:40:21
lat                    38.880452
lng                  -123.050671
duration_work    0 days 07:48:10
lat_work                38.43655
lng_work             -122.717506
dtype: object